# GFSM v1 Area Summary Example

This notebook calculates area by GFSM class from a local tile or clipped raster and writes a small CSV output. Use an equal-area CRS for rigorous area estimates.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import rasterio
from pyproj import CRS
from pyproj import Geod

tile_path = Path('path/to/GFSM_tile.tif')
output_csv = Path('gfsm_area_by_class.csv')

In [ ]:
class_names = {1: 'Very Low', 2: 'Low', 3: 'Moderate', 4: 'High', 5: 'Very High'}

def projected_pixel_area_km2(src):
    transform = src.transform
    area_in_crs_units = abs(transform.a * transform.e - transform.b * transform.d)
    x_factor = 1.0
    y_factor = 1.0
    if src.crs:
        axis_info = CRS.from_user_input(src.crs).axis_info
        if axis_info:
            x_factor = getattr(axis_info[0], 'unit_conversion_factor', 1.0)
        y_factor = getattr(axis_info[1], 'unit_conversion_factor', x_factor) if len(axis_info) > 1 else x_factor
    return area_in_crs_units * x_factor * y_factor / 1_000_000.0

def geodesic_area_by_class_km2(data, transform, nodata):
    if not np.isclose(transform.b, 0.0) or not np.isclose(transform.d, 0.0):
        raise ValueError('Rotated geographic rasters are not supported here; reproject to an equal-area CRS first.')
    geod = Geod(ellps='WGS84')
    areas = {value: 0.0 for value in class_names}
    for row in range(data.shape[0]):
        left, top = transform * (0, row)
        right, bottom = transform * (1, row + 1)
        pixel_area_m2, _ = geod.polygon_area_perimeter([left, right, right, left], [top, top, bottom, bottom])
        pixel_area_km2 = abs(pixel_area_m2) / 1_000_000.0
        row_values = data[row, :]
        for value in class_names:
            areas[value] += int(np.count_nonzero((row_values == value) & (row_values != nodata))) * pixel_area_km2
    return areas

with rasterio.open(tile_path) as src:
    if src.crs and src.crs.is_geographic:
        warnings.warn('Using WGS84 geodesic pixel areas; equal-area reprojection is recommended for production summaries.')
        data = src.read(1, masked=False)
        nodata = src.nodata if src.nodata is not None else 0
        area_by_class = geodesic_area_by_class_km2(data, src.transform, nodata)
    else:
        if not src.crs:
            warnings.warn('CRS is missing. Area is calculated from transform units as if they are meters.')
        data = src.read(1, masked=False)
        nodata = src.nodata if src.nodata is not None else 0
        pixel_area_km2 = projected_pixel_area_km2(src)
        area_by_class = {value: int(np.count_nonzero((data == value) & (data != nodata))) * pixel_area_km2 for value in class_names}

rows = []
for value, name in class_names.items():
    pixel_count = int(np.count_nonzero((data == value) & (data != nodata)))
    rows.append({'value': value, 'class_name': name, 'pixel_count': pixel_count, 'area_km2': area_by_class[value]})

area_table = pd.DataFrame(rows)
area_table

In [ ]:
area_table.to_csv(output_csv, index=False)
output_csv